# COMPASS multivariate_longitudinal models

Dynamic-DeepHit runs both platinum and NEPC at every landmark, each with a cause-only and a cause-plus-death competing-risk configuration. The endpoints read their independent prediction-input trees created by `01_preprocessing.ipynb`.

Optional cluster paths are configured by endpoint in `PREDICTION_INPUT_DIRS_BY_ENDPOINT`. SurvLatent ODE remains off by default and uses the same four configurations when enabled.

## Model arms

| toggle | model | predicts | inputs |
|---|---|---|---|
| *(always on)* | `dynamic-deephit` | one CIF per patient, at the landmark | `longitudinal_landmark{D}.csv` |
| `RUN_DYNAMIC` | `dynamic-deephit-dyn` | a CIF at **every** observation time, from history up to that time | `longitudinal_full_landmark{D}.csv` |
| `RUN_SURVLATENT` | `survlatent-ode` | continuous-time latent ODE | landmark inputs |

The landmark arm is the comparison of record: it is the row directly comparable to the Cox/XGBoost arms in `03_multivariate.ipynb`, and turning on the dynamic arm never changes it. The dynamic arm writes its headline `*_metrics.csv` at `prediction_time == landmark_time` so the two are read the same way and appear side by side in the summary table.

### To predict time-to-platinum on the full ADT cohort

```python
ARMS       = ["adt"]
ENDPOINTS  = ("platinum",)
COHORTS    = ("all",)          # no MRN restriction
EXCLUSIONS = ("none",)
RUN_DYNAMIC = True
```

`COHORTS = ("all",)` is what makes this the *full* arm cohort — the other cohort keys narrow it via `--restrict-to-mrns` upstream. The dynamic arm additionally requires inputs built with `--longitudinal-full-followup` (see the configuration cell).

In [ ]:
from pathlib import Path

ARMS = ["adt"]

# Time-to-platinum on the FULL ADT cohort. The commented values are the wider
# grid this notebook used previously -- restore them to go back to it.
ENDPOINTS = ("platinum",)          # was: ("platinum", "nepc")
COHORTS = ("all",)                 # full arm cohort, no MRN restriction
# COHORTS = ("all", "metastatic_adt", "metastatic_llm")
#   metastatic_adt -- retrospective ADT-intent stratum
#   metastatic_llm -- met_diagnosis LLM metastatic status
# Orthogonal to COHORTS: "none" keeps every patient, "pre_adt_castrate"
# drops those with a castrate testosterone (<50 ng/dL) before ADT start,
# who were presumably androgen-deprived elsewhere first.
EXCLUSIONS = ("none",)             # was: ("none", "pre_adt_castrate")
RUN_SURVLATENT = False

# True dynamic (per-timepoint) Dynamic-DeepHit: one CIF at EVERY observation
# time from history up to that time, instead of a single prediction at the
# landmark. Adds an arm -- the landmark arm still runs, and its row stays the
# one comparable to 03_multivariate.ipynb's Cox/XGBoost.
#
# PREREQUISITE: the prediction-input tree must have been built with
#   build_prediction_inputs.py --longitudinal-full-followup
# i.e. cp.BUILD_FULL_FOLLOWUP = True in 01_preprocessing.ipynb Stage 3, which
# writes longitudinal_full_landmark{D}.csv alongside the landmark files.
# Setting RUN_DYNAMIC here CANNOT create those files after the fact; without
# them the run raises FileNotFoundError naming the flag, rather than silently
# scoring the landmark frame twice.
#
# COST: this is 6 extra fits at the config above (3 landmarks x 2 configs), and
# per-timepoint prediction means many more rows per patient than the landmark
# arm. To smoke-test the arm first, narrow to one cell with
#   RUNS[0]["landmarks"] = [0]
# before running the next cell.
RUN_DYNAMIC = True

OVERWRITE = False  # True: refit and replace existing outputs; False: resume/skip

# Optional endpoint -> arm -> path overrides for GPU/shared-filesystem jobs.
PREDICTION_INPUT_DIRS_BY_ENDPOINT = {
    # "platinum": {"adt": Path("/cluster/path/to/prediction_inputs_adt")},
    # "nepc": {"adt": Path("/cluster/path/to/prediction_inputs_adt_nepc")},
}

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.MAX_PRED_WINDOW = 260
cp.RUN_SURVLATENT = RUN_SURVLATENT
cp.RUN_DYNAMIC = RUN_DYNAMIC
cp.FORCE_RERUN = OVERWRITE

if RUN_SURVLATENT:
    cp.SURVLATENT_REPO = cp.DEFAULT_SURVLATENT_REPO

RUNS = cp.make_endpoint_runs(
    ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=EXCLUSIONS,
    prediction_input_dirs_by_endpoint=PREDICTION_INPUT_DIRS_BY_ENDPOINT,
)
for endpoint in ENDPOINTS:
    specs = cp.longitudinal_task_specs(endpoint)
    models = sorted({m for m, _, _ in specs})
    print(f"endpoint={endpoint}  configs={sorted({c for _, c, _ in specs})}  models={models}")
print(f"\n{len(RUNS)} run(s): {[r['label'] for r in RUNS]}")

# The dynamic arm reads files the landmark arm does not. Check now rather than
# after the landmark fits have already run.
if RUN_DYNAMIC:
    for run in RUNS:
        missing = [
            str(run["inputs_dir"] / f"longitudinal_full_landmark{d}.csv")
            for d in run["landmarks"]
            if not (run["inputs_dir"] / f"longitudinal_full_landmark{d}.csv").exists()
        ]
        if missing:
            print(
                f"\n[warn] {run['label']}: RUN_DYNAMIC is on but full-follow-up inputs "
                f"are missing ({len(missing)} of {len(run['landmarks'])} landmarks), e.g.\n"
                f"        {missing[0]}\n"
                "        Set cp.BUILD_FULL_FOLLOWUP = True in 01_preprocessing.ipynb "
                "Stage 3 and re-run it."
            )

## Run multivariate_longitudinal models

Dynamic-DeepHit in both of the endpoint's configs — plus the per-timepoint
dynamic arm if `RUN_DYNAMIC` was set, and SurvLatent ODE if `RUN_SURVLATENT`
was. Set `OVERWRITE = True` in the configuration cell to refit and replace
existing outputs. With `False`, tasks whose metrics file already exists are
skipped, so an interrupted run resumes where it stopped. Layout:
`local_runs_<arm>[_nepc]/multivariate_longitudinal/<model>/landmark_{0,90,180}/<config>/`.

The two DeepHit arms write to sibling directories (`dynamic_deephit/` and
`dynamic_deephit_dynamic/`) and use distinct metrics filenames, so neither can
overwrite the other and each resumes independently.

In [ ]:
for run in RUNS:
    cp.run_multivariate_longitudinal(run)

## Summary tables

Per-run C-index / mean AUC(t) / integrated Brier for every (model, landmark,
config), filtered to the endpoint's cause-of-interest row for headline
comparability against `03_multivariate.ipynb`'s Cox/XGBoost arms, then
combined across runs. The competing configs' death rows are written to disk
but excluded here — read them from the metrics CSVs directly if needed.

In [ ]:
summary_dfs = {cp.run_key(run): cp.summarize_longitudinal_outputs(run) for run in RUNS}
for label, df in summary_dfs.items():
    print(f"=== {label} ===")
    display(df)

In [ ]:
import pandas as pd

combined_longitudinal_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_longitudinal_summary_df

## Incremental value of accruing lab history (dynamic arm only)

Written by `cp.run_incremental_risk`, which post-processes the dynamic arm's
per-timepoint predictions — no refit. Empty unless `RUN_DYNAMIC` was set.

**Lead with the ablation (5b).** At each prediction time it compares the model's
own prediction from full history against its prediction from history truncated
`delta` days earlier, on the *same* patients and the *same* horizon outcome. A
positive `auc_gain` is therefore attributable to the newest labs. This works
without a second fit because the GRU is causal: the prediction already emitted
at step `t - delta` *is* the estimate from history truncated there.

The sequential-landmark series (5a, `which="by_landmark"`) is reported too, but
its trend is confounded — the risk set shrinks as the prediction time grows and
is increasingly selected for patients who have not yet received platinum, so a
rising AUC there is not a clean "the model improves" claim.

Rows whose risk set is too small to support a metric carry NaN metrics and a
`note` beginning `underpowered:` rather than being dropped, so a gap in the
series shows why it is missing.

In [ ]:
incremental_ablation_df = pd.concat(
    [cp.load_incremental_risk_results(run, "ablation") for run in RUNS],
    ignore_index=True,
) if RUNS else pd.DataFrame()

if incremental_ablation_df.empty:
    print("No incremental-risk output (RUN_DYNAMIC off, or the dynamic arm has not run).")
else:
    display(
        incremental_ablation_df[
            [
                "run", "landmark_days", "config", "prediction_day",
                "n_scored", "n_events",
                "auc_held_back", "auc_full_history", "auc_gain",
                "brier_gain", "note",
            ]
        ]
    )

In [ ]:
# 5a: the confounded sequential-landmark series. Read alongside 5b, not instead.
incremental_by_landmark_df = pd.concat(
    [cp.load_incremental_risk_results(run, "by_landmark") for run in RUNS],
    ignore_index=True,
) if RUNS else pd.DataFrame()

if incremental_by_landmark_df.empty:
    print("No by-landmark output (RUN_DYNAMIC off, or the dynamic arm has not run).")
else:
    display(incremental_by_landmark_df)